# Pairs trading & cointegration

_Build a cointegrated pair, trade the z-score, and pay the spread._

**pyportfolios.com** · [/research/pairs-trading-cointegration](https://pyportfolios.com/research/pairs-trading-cointegration)

> Runs on numpy + pandas + scipy + matplotlib. Synthetic, seeded data — illustrative, not investment advice.

## Note

The article uses `statsmodels` (`adfuller`, OLS). Here we estimate the hedge ratio
with `numpy.polyfit` so the notebook runs without statsmodels; the ADF test is shown
as an optional cell.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(4)
n = 1500
common = np.cumsum(rng.normal(0, 1, n))           # shared stochastic trend
A = 50 + common + rng.normal(0, 1, n)
spread_true = rng.normal(0, 1, n)
spread_true = pd.Series(spread_true).ewm(span=10).mean().values * 3   # mean-reverting
B = 30 + 0.8 * common + spread_true               # B shares A's trend -> cointegrated
A = pd.Series(A); B = pd.Series(B)

beta = np.polyfit(np.log(B), np.log(A), 1)[0]     # hedge ratio
spread = np.log(A) - beta * np.log(B)
print(f"hedge ratio beta = {beta:.2f}")

## Z-score signal with entry/exit hysteresis, and a costed backtest

In [ ]:
window, entry, exit_ = 60, 2.0, 0.5
z = (spread - spread.rolling(window).mean()) / spread.rolling(window).std()

state = np.where(z < -entry, 1, np.where(z > entry, -1, np.nan))
pos = pd.Series(state, index=z.index)
pos[z.abs() < exit_] = 0
pos = pos.ffill().fillna(0)

pnl = pos.shift(1) * spread.diff()
net = pnl - pos.diff().abs().shift(1) * 0.0005     # 5 bps per leg change
sharpe = np.sqrt(252) * net.mean() / net.std()
print(f"net Sharpe {sharpe:.2f}   trades {int(pos.diff().abs().sum())}")

## Optional — the formal Engle–Granger test (needs statsmodels)

In [ ]:
try:
    from statsmodels.tsa.stattools import adfuller
    print(f"ADF p-value on the spread: {adfuller(spread.dropna())[1]:.4f}")
except ImportError:
    print("statsmodels not installed -> pip install statsmodels")